# QSilver: DFT y QFT

Este notebook desarrolla una exposición autoexplicativa sobre la Transformada Discreta de Fourier (DFT) y la Transformada Cuántica de Fourier (QFT). El objetivo no es memorizar una colección de fórmulas, sino entender por qué Fourier aparece de forma natural cuando la información está codificada en fases, periodicidades o frecuencias.

La notación de Dirac se escribe sin macros personalizadas para mantener compatibilidad con JupyterLab, Anaconda y Google Colab. Por ejemplo, se usará:

$$
\left|x\right\rangle,\qquad
\left|01\right\rangle,\qquad
\left\langle x\right|.
$$

La DFT se tratará como una transformación matricial sobre vectores complejos. La QFT se tratará como la misma matriz unitaria aplicada al vector de amplitudes de un estado cuántico.

## 1. Objetivos de aprendizaje

Al finalizar el notebook deberías poder formular la DFT unitaria, interpretar sus coeficientes como proyecciones sobre modos de frecuencia, visualizar raíces de la unidad, construir matrices de Fourier, comparar una DFT clásica con una QFT, derivar la acción de la QFT sobre estados base pequeños y construir circuitos QFT manualmente en Qiskit y Cirq.

También deberías poder explicar por qué la QFT no es una forma de imprimir todos los coeficientes de Fourier de un estado cuántico. La QFT aplica un cambio de base sobre amplitudes, pero la medición produce muestras; su valor aparece cuando se integra en un algoritmo que convierte fases o periodicidades en resultados observables.

## 2. Prerrequisitos operativos

Se asume que ya conoces números complejos, módulo, conjugado, forma polar, amplitudes complejas, regla de Born, matrices unitarias, fase global y fase relativa. Aquí esos conceptos se usan directamente.

La idea central del módulo es que las raíces de la unidad funcionan como patrones de comparación. Si una señal o un estado se alinea con un patrón de fase, la suma compleja se refuerza; si no se alinea, las contribuciones se cancelan.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi, sqrt

np.set_printoptions(precision=4, suppress=True)

def format_complex(z: complex, tol: float = 1e-10) -> str:
    """Representación breve de un número complejo para impresión."""
    a = 0.0 if abs(z.real) < tol else z.real
    b = 0.0 if abs(z.imag) < tol else z.imag
    return f"{a:.4g}{b:+.4g}i"

## 3. Raíces de la unidad

Para una longitud $N$, la raíz fundamental de la unidad es

$$
\omega_N=e^{2\pi i/N}.
$$

Sus potencias recorren $N$ puntos igualmente espaciados sobre el círculo unitario. La DFT usa esos puntos como fases de comparación. La geometría de estas raíces explica la cancelación de frecuencias incompatibles.

In [ ]:
def plot_roots_of_unity(N: int = 8) -> None:
    roots = np.exp(2j * np.pi * np.arange(N) / N)
    plt.figure(figsize=(6, 6))
    ax = plt.gca()
    ax.axhline(0, color="0.4", linewidth=1)
    ax.axvline(0, color="0.4", linewidth=1)
    circle = plt.Circle((0, 0), 1, fill=False, linestyle="--", color="0.5")
    ax.add_artist(circle)
    plt.scatter(roots.real, roots.imag, s=80)
    for k, z in enumerate(roots):
        plt.text(1.18 * z.real, 1.18 * z.imag, f"k={k}", ha="center", va="center")
    plt.title(f"Raíces {N}-ésimas de la unidad")
    plt.xlabel("Parte real")
    plt.ylabel("Parte imaginaria")
    plt.axis("equal")
    plt.grid(alpha=0.25)
    plt.show()

plot_roots_of_unity(8)

## 4. Definición unitaria de la DFT

Usaremos la convención unitaria de la DFT:

$$
X_k=\frac{1}{\sqrt{N}}\sum_{n=0}^{N-1}x_n e^{2\pi i kn/N}.
$$

Esta normalización es importante porque preserva la norma del vector. En computación cuántica, preservar norma no es opcional: una operación cerrada sobre estados debe ser unitaria.

In [ ]:
def dft_matrix(N: int, sign: int = +1) -> np.ndarray:
    """Matriz DFT unitaria de tamaño N.

    sign=+1 usa exp(+2πikn/N).
    sign=-1 usa exp(-2πikn/N).
    """
    k = np.arange(N).reshape((N, 1))
    n = np.arange(N).reshape((1, N))
    return np.exp(sign * 2j * np.pi * k * n / N) / np.sqrt(N)


def dft(x: np.ndarray, sign: int = +1) -> np.ndarray:
    """Aplica la DFT unitaria a un vector complejo."""
    x = np.asarray(x, dtype=complex)
    return dft_matrix(len(x), sign=sign) @ x

# Verificación de unitariedad para N=8
N = 8
F = dft_matrix(N)
print("¿F†F ≈ I?", np.allclose(F.conj().T @ F, np.eye(N)))

## 5. Visualización de la matriz DFT

La matriz DFT tiene entradas de igual magnitud y fases organizadas de manera regular. La magnitud constante indica que la información no se amplifica ni se atenúa por entrada individual; la estructura real está en las fases.

In [ ]:
def plot_dft_matrix(N: int = 8) -> None:
    F = dft_matrix(N)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    im0 = axes[0].imshow(np.abs(F), cmap="viridis")
    axes[0].set_title("Módulo de la matriz DFT")
    axes[0].set_xlabel("n")
    axes[0].set_ylabel("k")
    plt.colorbar(im0, ax=axes[0], fraction=0.046)

    im1 = axes[1].imshow(np.angle(F), cmap="twilight", vmin=-np.pi, vmax=np.pi)
    axes[1].set_title("Fase de la matriz DFT")
    axes[1].set_xlabel("n")
    axes[1].set_ylabel("k")
    plt.colorbar(im1, ax=axes[1], fraction=0.046)

    plt.tight_layout()
    plt.show()

plot_dft_matrix(8)

## 6. Ejemplo desarrollado: DFT de dos componentes

Para $N=2$, la DFT unitaria es

$$
F_2=\frac{1}{\sqrt{2}}
\begin{pmatrix}
1&1\\
1&-1
\end{pmatrix}.
$$

Aplicada al vector $(3,4)^T$, separa el modo uniforme y el modo alternante:

$$
F_2\begin{pmatrix}3\\4\end{pmatrix}
=
\begin{pmatrix}
7/\sqrt{2}\\
-1/\sqrt{2}
\end{pmatrix}.
$$

El primer coeficiente mide suma coherente; el segundo mide contraste.

In [ ]:
x = np.array([3, 4], dtype=complex)
X = dft(x)
print("Entrada:", x)
print("DFT unitaria:", X)

plt.figure(figsize=(6, 4))
plt.bar(["x0", "x1"], x.real, alpha=0.7, label="entrada")
plt.bar(["X0", "X1"], X.real, alpha=0.7, label="salida DFT")
plt.title("DFT unitaria de (3, 4)")
plt.ylabel("Valor real")
plt.grid(axis="y", alpha=0.3)
plt.legend()
plt.show()

## 7. Componentes armónicas discretas

Las filas de la DFT pueden verse como detectores de frecuencia. En la siguiente figura se muestran partes reales de distintos modos armónicos. Cada modo tiene una velocidad de oscilación distinta; la DFT mide cuánto de cada modo aparece en la señal.

In [ ]:
def plot_fourier_modes(N: int = 32, modes=(0, 1, 3, 8)) -> None:
    n = np.arange(N)
    plt.figure(figsize=(10, 4.5))
    for k in modes:
        mode = np.exp(2j * np.pi * k * n / N)
        plt.plot(n, mode.real, marker="o", markersize=3, label=f"k={k}")
    plt.title("Modos de Fourier: parte real")
    plt.xlabel("Índice n")
    plt.ylabel("Re(exp(2πikn/N))")
    plt.grid(alpha=0.25)
    plt.legend(ncol=len(modes))
    plt.show()

plot_fourier_modes()

## 8. Señales y espectros

Ahora construiremos una señal discreta formada por dos componentes. En el dominio original se observa una secuencia de valores; en el dominio de Fourier aparecen picos que identifican las frecuencias dominantes.

In [ ]:
N = 64
n = np.arange(N)
signal = np.sin(2 * np.pi * 5 * n / N) + 0.6 * np.sin(2 * np.pi * 14 * n / N + 0.4)
spectrum = dft(signal)

plt.figure(figsize=(10, 4))
plt.plot(n, signal, marker="o", markersize=3)
plt.title("Señal discreta")
plt.xlabel("n")
plt.ylabel("x[n]")
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(10, 4))
plt.stem(np.arange(N // 2), np.abs(spectrum[:N // 2]))
plt.title("Magnitud de la DFT")
plt.xlabel("k")
plt.ylabel("|X[k]|")
plt.grid(alpha=0.25)
plt.show()

## 9. Suma geométrica de fases

La cancelación de Fourier puede visualizarse como una suma de flechas en el plano complejo. Si las flechas rodean el círculo de forma equilibrada, la suma es pequeña o nula. Si se alinean, la suma es grande.

In [ ]:
def plot_phase_sum(N: int = 12, frequency: int = 3) -> None:
    phases = np.exp(2j * np.pi * frequency * np.arange(N) / N)
    plt.figure(figsize=(6, 6))
    ax = plt.gca()
    ax.axhline(0, color="0.4", linewidth=1)
    ax.axvline(0, color="0.4", linewidth=1)
    for z in phases:
        plt.arrow(0, 0, z.real, z.imag, head_width=0.03, length_includes_head=True, alpha=0.55)
    plt.scatter(phases.real, phases.imag, s=40)
    plt.title(f"Suma de fases para frecuencia {frequency}")
    plt.xlabel("Re")
    plt.ylabel("Im")
    plt.axis("equal")
    plt.grid(alpha=0.25)
    plt.show()
    print("Suma:", format_complex(np.sum(phases)))

plot_phase_sum(12, 3)

## 10. QFT: misma matriz, distinta operación

La QFT usa la matriz DFT unitaria, pero la aplica al vector de amplitudes de un estado cuántico. Para $N=2^n$:

$$
\mathcal{F}_N\left|x\right\rangle
=
\frac{1}{\sqrt{N}}\sum_{y=0}^{N-1}e^{2\pi ixy/N}\left|y\right\rangle.
$$

La salida es un estado cuántico. Una medición no devuelve todos los coeficientes; devuelve una muestra de la distribución resultante.

In [ ]:
def qft_matrix(n_qubits: int, sign: int = +1) -> np.ndarray:
    """Matriz QFT para n_qubits, usando N=2**n_qubits."""
    return dft_matrix(2 ** n_qubits, sign=sign)


def basis_state(index: int, n_qubits: int) -> np.ndarray:
    """Vector de estado base computacional."""
    vec = np.zeros(2 ** n_qubits, dtype=complex)
    vec[index] = 1.0
    return vec

nq = 2
F4 = qft_matrix(nq)
print("Matriz QFT para 2 qubits:")
print(F4)

## 11. Ejemplo desarrollado: QFT de $\left|01\right\rangle$

En dos qubits, $\left|01\right\rangle$ corresponde al índice $1$. La QFT produce:

$$
\mathcal{F}_4\left|01\right\rangle
=
\frac{1}{2}\left(
\left|00\right\rangle+i\left|01\right\rangle
-\left|10\right\rangle-i\left|11\right\rangle
\right).
$$

Todas las probabilidades son iguales, pero la fase cambia de componente en componente.

In [ ]:
psi_01 = basis_state(1, 2)
out_01 = F4 @ psi_01
labels_2q = ["00", "01", "10", "11"]
for label, amp in zip(labels_2q, out_01):
    print(f"|{label}>: {format_complex(amp)}")

plt.figure(figsize=(7, 4))
plt.bar(labels_2q, np.abs(out_01) ** 2)
plt.title("Probabilidades tras QFT de |01>")
plt.xlabel("Estado base")
plt.ylabel("Probabilidad")
plt.ylim(0, 0.35)
plt.grid(axis="y", alpha=0.25)
plt.show()

plt.figure(figsize=(7, 4))
plt.stem(range(4), np.angle(out_01))
plt.xticks(range(4), labels_2q)
plt.title("Fases de las amplitudes tras QFT de |01>")
plt.xlabel("Estado base")
plt.ylabel("Fase")
plt.grid(alpha=0.25)
plt.show()

## 12. Paisajes de fase de estados base

La QFT de un estado base produce amplitudes de igual módulo y fases con pendiente regular. Esa pendiente depende de la etiqueta de entrada. La información queda codificada como un gradiente de fase.

In [ ]:
def plot_qft_phase_landscapes(n_qubits: int = 3, xs=(1, 2, 3)) -> None:
    N = 2 ** n_qubits
    y = np.arange(N)
    fig, axes = plt.subplots(len(xs), 1, figsize=(9, 2.5 * len(xs)), sharex=True)
    if len(xs) == 1:
        axes = [axes]
    for ax, x in zip(axes, xs):
        state = qft_matrix(n_qubits) @ basis_state(x, n_qubits)
        ax.stem(y, np.angle(state))
        ax.set_ylabel(f"x={x}")
        ax.grid(alpha=0.25)
    axes[-1].set_xlabel("Etiqueta de salida y")
    fig.suptitle("Fases en QFT de estados base")
    plt.tight_layout()
    plt.show()

plot_qft_phase_landscapes()

## 13. Interferencia en la QFT

La QFT puede concentrar amplitud cuando las fases de entrada contienen periodicidad. La figura siguiente muestra la interferencia de dos componentes armónicas; la DFT permite separarlas porque cada componente se alinea con un detector distinto.

In [ ]:
N = 32
n = np.arange(N)
wave_1 = np.exp(2j * np.pi * 3 * n / N)
wave_2 = np.exp(2j * np.pi * 8 * n / N)
combined = wave_1 + wave_2

plt.figure(figsize=(10, 4))
plt.plot(n, combined.real, label="suma")
plt.plot(n, wave_1.real, linestyle="--", alpha=0.6, label="k=3")
plt.plot(n, wave_2.real, linestyle="--", alpha=0.6, label="k=8")
plt.title("Interferencia de componentes armónicas")
plt.xlabel("n")
plt.ylabel("parte real")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## 14. Compuertas de fase controlada en QFT

La QFT no se construye solo con Hadamards. Para tamaños mayores a dos se necesitan fases fraccionarias condicionadas. La familia $CR_k$ aplica una fase de ángulo

$$
\theta_k=\frac{2\pi}{2^k}.
$$

Estas fases construyen, bit a bit, el gradiente de Fourier.

In [ ]:
def crk_angle(k: int) -> float:
    """Ángulo de fase de CR_k."""
    return 2 * np.pi / (2 ** k)

ks = np.arange(2, 9)
angles = np.array([crk_angle(k) for k in ks])
plt.figure(figsize=(8, 4))
plt.stem(ks, angles / np.pi)
plt.xlabel("k")
plt.ylabel("Ángulo / π")
plt.title("Ángulos de fase controlada CR_k")
plt.xticks(ks)
plt.grid(alpha=0.25)
plt.show()

## 15. Acción de $CR_2$ sobre una superposición

Si $CR_2$ aplica fase $e^{i\pi/2}=i$ al componente $\left|11\right\rangle$, entonces

$$
\frac{\left|01\right\rangle+\left|11\right\rangle}{\sqrt{2}}
\mapsto
\frac{\left|01\right\rangle+i\left|11\right\rangle}{\sqrt{2}}.
$$

Las probabilidades de medir $01$ o $11$ no cambian, pero la fase relativa sí. Esa diferencia puede ser convertida en probabilidad por interferencia posterior.

In [ ]:
state = np.zeros(4, dtype=complex)
state[1] = 1 / np.sqrt(2)  # |01>
state[3] = 1 / np.sqrt(2)  # |11>

CR2 = np.eye(4, dtype=complex)
CR2[3, 3] = np.exp(1j * np.pi / 2)  # fase sobre |11>
new_state = CR2 @ state

for label, amp in zip(labels_2q, new_state):
    if abs(amp) > 1e-12:
        print(f"|{label}>: {format_complex(amp)}")

## 16. Circuito conceptual de la QFT

El circuito QFT procesa qubits de forma incremental. En cada etapa se aplica Hadamard y luego compuertas de fase controlada. Al final, con frecuencia se aplican SWAPs para corregir el orden de lectura.

In [ ]:
# Dibujo esquemático manual con Matplotlib.
plt.figure(figsize=(10, 4))
plt.axis("off")
y_positions = [0.75, 0.5, 0.25]
for i, y in enumerate(y_positions):
    plt.plot([0.05, 0.95], [y, y], color="black", linewidth=1.5)
    plt.text(0.02, y, f"q{i}", ha="right", va="center")

def box(x, y, label):
    plt.text(x, y, label, ha="center", va="center",
             bbox=dict(boxstyle="square", fc="#EEF2F7", ec="#2F4056"))

box(0.18, y_positions[0], "H")
box(0.34, y_positions[1], "R2")
plt.plot([0.34, 0.34], [y_positions[0], y_positions[1]], color="black")
plt.scatter([0.34], [y_positions[0]], color="black", s=25)
box(0.48, y_positions[2], "R3")
plt.plot([0.48, 0.48], [y_positions[0], y_positions[2]], color="black")
plt.scatter([0.48], [y_positions[0]], color="black", s=25)
box(0.62, y_positions[1], "H")
box(0.76, y_positions[2], "R2")
plt.plot([0.76, 0.76], [y_positions[1], y_positions[2]], color="black")
plt.scatter([0.76], [y_positions[1]], color="black", s=25)
box(0.86, y_positions[2], "H")
plt.text(0.92, 0.5, "SWAP", ha="center", va="center",
         bbox=dict(boxstyle="round", fc="#FCEFC8", ec="#2F4056"))
plt.title("Esquema conceptual de QFT de tres qubits")
plt.show()

## 17. QFT manual en NumPy

Antes de usar frameworks cuánticos, conviene verificar el operador con matrices. La siguiente función aplica QFT a cualquier estado de $n$ qubits representado como vector NumPy.

In [ ]:
def apply_qft_statevector(state: np.ndarray, sign: int = +1) -> np.ndarray:
    """Aplica QFT a un vector de estado cuya longitud es potencia de dos."""
    state = np.asarray(state, dtype=complex)
    N = len(state)
    if N & (N - 1) != 0:
        raise ValueError("La longitud debe ser potencia de dos")
    return dft_matrix(N, sign=sign) @ state

# Verificación: QFT de |01>
state_01 = basis_state(1, 2)
print(apply_qft_statevector(state_01))

## 18. Qiskit: instalación condicional

La siguiente celda intenta importar Qiskit. Si no está disponible, intenta instalarlo. En Google Colab esto permite ejecutar el notebook sin editar manualmente el entorno. En instalaciones locales administradas, puedes omitir la instalación si Qiskit ya está disponible.

In [ ]:
import importlib
import subprocess
import sys


def ensure_package(import_name: str, pip_name: str | None = None) -> bool:
    """Importa un paquete; si no existe, intenta instalarlo con pip."""
    try:
        importlib.import_module(import_name)
        return True
    except ImportError:
        if pip_name is None:
            pip_name = import_name
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name, "-q"])
            importlib.import_module(import_name)
            return True
        except Exception as exc:
            print(f"No se pudo instalar {pip_name}: {exc}")
            return False

qiskit_available = ensure_package("qiskit", "qiskit")
qiskit_aer_available = ensure_package("qiskit_aer", "qiskit-aer")
print("Qiskit disponible:", qiskit_available)
print("Qiskit Aer disponible:", qiskit_aer_available)

## 19. QFT manual en Qiskit

La implementación manual aplica Hadamards, fases controladas y SWAPs. Construirla explícitamente permite inspeccionar la estructura que una función de biblioteca podría ocultar.

In [ ]:
if qiskit_available:
    from qiskit import QuantumCircuit
    try:
        from qiskit.quantum_info import Operator, Statevector
    except Exception as exc:
        print("Qiskit está instalado, pero no se pudieron importar Operator/Statevector:", exc)


def qft_qiskit(n: int, do_swaps: bool = True):
    """Circuito QFT manual en Qiskit."""
    if not qiskit_available:
        print("Qiskit no está disponible en este entorno.")
        return None
    qc = QuantumCircuit(n, name="QFT")
    for j in range(n):
        qc.h(j)
        for m in range(j + 1, n):
            angle = np.pi / (2 ** (m - j))
            qc.cp(angle, m, j)
    if do_swaps:
        for j in range(n // 2):
            qc.swap(j, n - 1 - j)
    return qc

qc_qft3 = qft_qiskit(3)
if qc_qft3 is not None:
    display(qc_qft3.draw("text"))

## 20. Comparación entre matriz QFT y circuito Qiskit

Para pocos qubits podemos comparar la matriz de QFT construida con NumPy y la matriz del circuito. Dependiendo de la convención de orden de qubits, puede ser necesario considerar inversión de bits; por eso el análisis de orden es parte de la verificación.

In [ ]:
if qiskit_available and qc_qft3 is not None:
    U_qiskit = Operator(qc_qft3).data
    print("Matriz del circuito Qiskit, forma:", U_qiskit.shape)
    print("La matriz es unitaria:", np.allclose(U_qiskit.conj().T @ U_qiskit, np.eye(8)))
else:
    print("Se omite comparación porque Qiskit no está disponible.")

## 21. Cirq: instalación condicional

La construcción en Cirq usa una idea equivalente. Las compuertas controladas de fase pueden expresarse con `CZPowGate`, cuyo exponente controla la fase aplicada al componente activado.

In [ ]:
cirq_available = ensure_package("cirq", "cirq")
print("Cirq disponible:", cirq_available)

## 22. QFT manual en Cirq

En Cirq, una fase controlada compatible con $CR_k$ puede construirse mediante `cirq.CZPowGate(exponent=2/2**k)`. El exponente determina la fase sobre el componente controlado.

In [ ]:
if cirq_available:
    import cirq


def qft_cirq(n: int, do_swaps: bool = True):
    """Circuito QFT manual en Cirq."""
    if not cirq_available:
        print("Cirq no está disponible en este entorno.")
        return None
    qubits = cirq.LineQubit.range(n)
    circuit = cirq.Circuit()
    for j in range(n):
        circuit.append(cirq.H(qubits[j]))
        for m in range(j + 1, n):
            k = m - j + 1
            gate = cirq.CZPowGate(exponent=2 / (2 ** k))
            circuit.append(gate.on(qubits[m], qubits[j]))
    if do_swaps:
        for j in range(n // 2):
            circuit.append(cirq.SWAP(qubits[j], qubits[n - 1 - j]))
    return circuit

qc_cirq3 = qft_cirq(3)
if qc_cirq3 is not None:
    print(qc_cirq3)

## 23. Visualización de $CR_k$ como fase fraccionaria

La jerarquía $CR_2,CR_3,CR_4,\ldots$ introduce rotaciones cada vez más pequeñas. Estas fases finas son necesarias para reconstruir la fase total de Fourier en registros con varios qubits.

In [ ]:
plt.figure(figsize=(8, 4))
ks = np.arange(2, 10)
angles = [crk_angle(k) for k in ks]
plt.plot(ks, np.array(angles) / np.pi, marker="o")
plt.xlabel("k")
plt.ylabel("ángulo / π")
plt.title("Jerarquía de fases en CR_k")
plt.grid(alpha=0.25)
plt.show()

for k, a in zip(ks, angles):
    print(f"CR_{k}: ángulo = {a/np.pi:.6f}π")

## 24. DFT vs QFT: comparación visual

La DFT produce un vector clásico de coeficientes. La QFT produce un estado cuántico. La matriz es la misma, pero el acceso a la información es distinto: el estado cuántico debe medirse, y la medición no entrega todos los coeficientes simultáneamente.

In [ ]:
plt.figure(figsize=(9, 4))
plt.axis("off")

def label_box(x, y, text):
    plt.text(x, y, text, ha="center", va="center",
             bbox=dict(boxstyle="round", fc="#EEF2F7", ec="#2F4056"))

label_box(0.15, 0.72, "Vector clásico x")
label_box(0.50, 0.72, "Matriz DFT F_N")
label_box(0.85, 0.72, "Coeficientes X")
plt.annotate("", xy=(0.38, 0.72), xytext=(0.25, 0.72), arrowprops=dict(arrowstyle="->"))
plt.annotate("", xy=(0.73, 0.72), xytext=(0.60, 0.72), arrowprops=dict(arrowstyle="->"))

label_box(0.15, 0.30, "Estado cuántico")
label_box(0.50, 0.30, "Operador QFT")
label_box(0.85, 0.30, "Estado transformado")
plt.annotate("", xy=(0.38, 0.30), xytext=(0.25, 0.30), arrowprops=dict(arrowstyle="->"))
plt.annotate("", xy=(0.73, 0.30), xytext=(0.60, 0.30), arrowprops=dict(arrowstyle="->"))
plt.title("DFT y QFT: misma matriz, distinta lectura operacional")
plt.show()

## 25. Ejercicios guiados

### Ejercicio 1: DFT de dos componentes

Calcular e interpretar la DFT unitaria de $(5,1)^T$.

El primer coeficiente representa el modo uniforme:

$$
(5+1)/\sqrt{2}=6/\sqrt{2}.
$$

El segundo representa el contraste alternante:

$$
(5-1)/\sqrt{2}=4/\sqrt{2}.
$$

La salida no debe interpretarse como dos valores nuevos sin contexto; representa la descomposición del vector en modos de Fourier.

### Ejercicio 2: QFT de $\left|10\right\rangle$

Para dos qubits, $\left|10\right\rangle$ corresponde a $x=2$. La fase de salida es

$$
e^{2\pi i(2)y/4}=e^{\pi i y}.
$$

Por tanto las fases alternan $1,-1,1,-1$ y la salida es

$$
\frac{1}{2}\left(
\left|00\right\rangle-
\left|01\right\rangle+
\left|10\right\rangle-
\left|11\right\rangle
\right).
$$

La información de la etiqueta se codifica en el patrón de fase, no en las magnitudes.

### Ejercicio 3: fase de $CR_4$

La familia $CR_k$ usa el ángulo

$$
\theta_k=\frac{2\pi}{2^k}.
$$

Para $k=4$:

$$
\theta_4=\frac{2\pi}{16}=\frac{\pi}{8}.
$$

Esta fase corresponde a una corrección fina. En la QFT, las fases pequeñas codifican contribuciones de bits menos significativos.

## 26. Ejercicios propuestos

1. Calcula la DFT unitaria de $(1,0,0,0)^T$ e interpreta el resultado como distribución uniforme de fases.
2. Calcula la QFT de $\left|11\right\rangle$ para dos qubits y explica el patrón de fases.
3. Construye la matriz DFT unitaria de tamaño $4$ y verifica numéricamente que es unitaria.
4. Modifica el circuito QFT en Qiskit para omitir los SWAPs y compara la matriz resultante.
5. En Cirq, cambia el exponente de una compuerta controlada de fase y observa cómo cambia la matriz del circuito.

## 27. Resumen final

La DFT es un cambio de base que expresa un vector discreto en componentes de frecuencia. Su estructura se basa en raíces de la unidad, sumas de fases y ortogonalidad de modos.

La QFT aplica la misma matriz unitaria al vector de amplitudes de un estado cuántico. Su circuito evita construir una matriz exponencialmente grande mediante Hadamards, fases controladas y SWAPs.

La QFT no debe entenderse como un método para imprimir todos los coeficientes de Fourier de un estado. Su valor aparece como subrutina dentro de algoritmos que codifican información en fases o periodicidad y luego usan interferencia para hacerla observable.